# Case Study: Sentiment Analysis

### Data Prep

In [2]:
import pandas as pd
import numpy as np

# Read in the data
df = pd.read_csv('Reviews.csv')

# Sample the data to speed up computation
# Comment out this line to match with lecture
df = df.sample(frac=0.1, random_state=10)

df.head()

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
160375,160376,B001EO5SW6,A1CNMOTETPJVC7,Stephan J. Gold,1,1,4,1242000000,Enjoyable,Very good for digestive system. Easy to eat wi...
92654,92655,B001AHJ2D8,AT9U5ZE5OO84C,"M. Kooiman ""nerdette""",5,5,5,1226534400,A tasty snack for the health conscious,"I am a label reader - I count fat grams, carbs..."
367563,367564,B003B3OOPA,A7QZ3NG2GL2CL,Autumn,3,3,4,1329696000,Delicious and a world of uses!,"Before I get rambling, I have to say this stuf..."
557346,557347,B0014C2JFC,A22F6OPV6XKJ0K,"David H. Kuehn ""Consulting, inc.""",2,3,5,1227398400,Watermellon- Jolly Ranchers,Jolly Ranchers Watermellon hard candy from Ama...
143941,143942,B000I1RGQC,A2KZRK2T9KFA0X,Sarah,1,1,5,1332460800,Much better then I anticipated,These are much better then I thought they woul...


In [3]:
# Drop missing values
df.dropna(inplace=True)

# Remove any 'neutral' ratings equal to 3
df = df[df['Score'] != 3]

# Encode 4s and 5s as 1 (rated positively)
# Encode 1s and 2s as 0 (rated poorly)
df['Positively Rated'] = np.where(df['Score'] > 3, 1, 0)
df.head(10)

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text,Positively Rated
160375,160376,B001EO5SW6,A1CNMOTETPJVC7,Stephan J. Gold,1,1,4,1242000000,Enjoyable,Very good for digestive system. Easy to eat wi...,1
92654,92655,B001AHJ2D8,AT9U5ZE5OO84C,"M. Kooiman ""nerdette""",5,5,5,1226534400,A tasty snack for the health conscious,"I am a label reader - I count fat grams, carbs...",1
367563,367564,B003B3OOPA,A7QZ3NG2GL2CL,Autumn,3,3,4,1329696000,Delicious and a world of uses!,"Before I get rambling, I have to say this stuf...",1
557346,557347,B0014C2JFC,A22F6OPV6XKJ0K,"David H. Kuehn ""Consulting, inc.""",2,3,5,1227398400,Watermellon- Jolly Ranchers,Jolly Ranchers Watermellon hard candy from Ama...,1
143941,143942,B000I1RGQC,A2KZRK2T9KFA0X,Sarah,1,1,5,1332460800,Much better then I anticipated,These are much better then I thought they woul...,1
29414,29415,B000PDY3P0,AQLFJMVN79I7V,flashblub,0,0,5,1324166400,Great popcorn,"Tastes just like theater pop corn, great flavo...",1
227365,227366,B003OB6CGI,A2AWPUDT68ACW6,Lori,1,1,5,1335571200,excellent,This product is excellent for so many things. ...,1
511703,511704,B001SB864M,A1FM2Z84TA9HH9,"ISO Health, Wealth, and Amazing Shopping on A...",10,11,5,1263600000,Delicious Alternative to Regular Potato Chips,The Good Health Humbles product line of their ...,1
163494,163495,B001UJIFRU,AAJ93RPWNYA89,nancy,2,2,1,1285027200,Dont like the taste,I love all of the Vita Coco's except this one....,0
554368,554369,B001EO7OAU,A3BH49ZKESHDID,Fred Camfield,2,2,5,1266192000,Something for your party or reception,I have sometimes ordered this from SeaBear for...,1


In [4]:
# Most ratings are positive
df['Positively Rated'].mean()

np.float64(0.8429450649646684)

In [5]:
from sklearn.model_selection import train_test_split

# Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(df['Text'],
                                                    df['Positively Rated'],
                                                    random_state=0)

In [6]:
print('X_train first entry:\n\n', X_train.iloc[0])
print('\n\nX_train shape: ', X_train.shape)

X_train first entry:

 These are very tasty sauces, and I am a heavy user for the great flavors. Perhaps I got used to the heat because to me they were not more than "medium" if regular Tabasco is "hot". I will buy them again for the flavor however.


X_train shape:  (39483,)


# CountVectorizer

In [7]:
from sklearn.feature_extraction.text import CountVectorizer

# Fit the CountVectorizer to the training data
vect = CountVectorizer().fit(X_train)

In [9]:
vect.get_feature_names_out()[::2000]

array(['00', 'alginate', 'b0038b1et4', 'brunvand', 'cluster', 'decant',
       'effecting', 'florentines', 'gum', 'inkling', 'likeable',
       'moderating', 'os', 'possibilty', 'reminders', 'shaft', 'steps',
       'timeline', 'vinegarish'], dtype=object)

In [11]:
len(vect.get_feature_names_out())

37675

In [13]:
# transform the documents in the training data to a document-term matrix
X_train_vectorized = vect.transform(X_train)

X_train_vectorized

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 2083298 stored elements and shape (39483, 37675)>

In [16]:
from sklearn.linear_model import LogisticRegression

# Train the model
model = LogisticRegression(max_iter=1000)
model.fit(X_train_vectorized, y_train)

LogisticRegression(max_iter=1000)

In [18]:
from sklearn.metrics import roc_auc_score

# Predict the transformed test documents
predictions = model.predict_proba(vect.transform(X_test))

print('AUC: ', roc_auc_score(y_test, predictions[:, 1]))

AUC:  0.9341814091566627


In [20]:
# get the feature names as numpy array
feature_names = np.array(vect.get_feature_names_out())

# Sort the coefficients from the model
sorted_coef_index = model.coef_[0].argsort()

# Find the 10 smallest and 10 largest coefficients
# The 10 largest coefficients are being indexed using [:-11:-1]
# so the list returned is in order of largest to smallest
print('Smallest Coefs:\n{}\n'.format(feature_names[sorted_coef_index[:10]]))
print('Largest Coefs: \n{}'.format(feature_names[sorted_coef_index[:-11:-1]]))

Smallest Coefs:
['sounded' 'worst' 'disappointment' 'disappointing' 'terrible'
 'flavorless' 'useless' 'bland' 'nasty' 'disgusting']

Largest Coefs: 
['perfect' 'delicious' 'pleased' 'beat' 'excellent' 'awesome' 'pleasantly'
 'wonderful' 'hooked' 'nicely']


# Tfidf

In [22]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Fit the TfidfVectorizer to the training data specifiying a minimum document frequency of 5
vect = TfidfVectorizer(min_df=5).fit(X_train)
len(vect.get_feature_names_out())

12055

In [24]:
X_train_vectorized = vect.transform(X_train)

model = LogisticRegression()
model.fit(X_train_vectorized, y_train)

predictions = model.predict_proba(vect.transform(X_test))

print('AUC: ', roc_auc_score(y_test, predictions[:, 1]))

AUC:  0.9504630438581163


In [26]:
feature_names = np.array(vect.get_feature_names_out())

sorted_tfidf_index = X_train_vectorized.max(0).toarray()[0].argsort()

print('Smallest tfidf:\n{}\n'.format(feature_names[sorted_tfidf_index[:10]]))
print('Largest tfidf: \n{}'.format(feature_names[sorted_tfidf_index[:-11:-1]]))

Smallest tfidf:
['parsing' 'buffoon' 'regrettable' 'unmeasured' '4thd' 'nations' '350mgs'
 '300mgs' 'fnb' 'academy']

Largest tfidf: 
['yum' 'nom' 'blech' 'br' 'mustard' 'breadsticks' 'salt' 'nutella' 'com'
 'chestnuts']


In [27]:
sorted_coef_index = model.coef_[0].argsort()

print('Smallest Coefs:\n{}\n'.format(feature_names[sorted_coef_index[:10]]))
print('Largest Coefs: \n{}'.format(feature_names[sorted_coef_index[:-11:-1]]))

Smallest Coefs:
['not' 'worst' 'disappointed' 'disappointing' 'terrible' 'unfortunately'
 'money' 'weak' 'awful' 'return']

Largest Coefs: 
['great' 'best' 'delicious' 'perfect' 'love' 'good' 'excellent'
 'wonderful' 'nice' 'loves']


In [41]:
# These reviews are treated the same by our current model
print(model.predict(vect.transform(['not an issue, phone is working',
                                    'an issue, phone is not working','product is not good'])))

[1 1 0]


# n-grams

In [34]:
# Fit the CountVectorizer to the training data specifiying a minimum
# document frequency of 5 and extracting 1-grams and 2-grams
vect = CountVectorizer(min_df=5, ngram_range=(1,2)).fit(X_train)

X_train_vectorized = vect.transform(X_train)

len(vect.get_feature_names_out())

84762

In [36]:
model = LogisticRegression()
model.fit(X_train_vectorized, y_train)

predictions = model.predict_proba(vect.transform(X_test))

print('AUC: ', roc_auc_score(y_test, predictions[:, 1]))

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


AUC:  0.9548682504704622


In [38]:
feature_names = np.array(vect.get_feature_names_out())

sorted_coef_index = model.coef_[0].argsort()

print('Smallest Coefs:\n{}\n'.format(feature_names[sorted_coef_index[:10]]))
print('Largest Coefs: \n{}'.format(feature_names[sorted_coef_index[:-11:-1]]))

Smallest Coefs:
['worst' 'terrible' 'disappointment' 'not worth' 'disappointing'
 'very disappointed' 'not good' 'disappointed' 'weak' 'nasty']

Largest Coefs: 
['delicious' 'not too' 'excellent' 'awesome' 'wonderful' 'perfect'
 'not bad' 'you won' 'great' 'love this']


In [39]:
# These reviews are now correctly identified
print(model.predict(vect.transform(['not an issue, phone is working',
                                    'an issue, phone is not working'])))

[1 1]
